# **Capstone Project : Bike Rental Dataset**

# **Problem Statements -**

Task 1:- Prepare a complete data analysis report on the given data.

Task 2:- Prediction of daily bike rental count based on the environmental and seasonal settings.


# **Importing the necessary libraries**

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns     # For visualization
import matplotlib.pyplot as plt     # For visualization
import warnings
warnings.filterwarnings('ignore')


# **Load Bike Rental Dataset**

In [2]:
# importing the dataset
df_day = pd.read_csv(r"C:\Users\Ramchandra Wadkar\OneDrive\Documents\Data_Science\CDS\Project\Capstone Projects\Bike Rental Dataset\day.csv")
df_day.head()

In [3]:
df_hour = pd.read_csv(r"C:\Users\Ramchandra Wadkar\OneDrive\Documents\Data_Science\CDS\Project\Capstone Projects\Bike Rental Dataset\hour.csv")
df_hour.head()               

#### **About the Dataset**

Both the files hour.csv and day.csv have the following fields, except hr which is not available in day.csv
- instant: record index
- dteday : date
- season : season (1:winter, 2:spring, 3:summer, 4:fall)
- yr : year (0: 2011, 1:2012)
- mnth : month ( 1 to 12)
- hr : hour (0 to 23)
- holiday : weather day is holiday or not (extracted from [Web Link])
- weekday : day of the week
- workingday : if neither weekend nor holiday, it is 1; otherwise 0.
- weathersit :
- 1: Clear, Few clouds, Partly cloudy, Partly cloudy
- 2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist
- 3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds
- 4: Heavy Rain + Ice Pellets + Thunderstorm + Mist, Snow + Fog
- temp : Normalized temperature in Celsius. The values are derived via (t-t_min)/(t_max-t_min), t_min=-8, t_max=+39 (only in hourly scale)
- atemp: Normalized feeling temperature in Celsius. The values are derived via (t-t_min)/(t_max-t_min), t_min=-16, t_max=+50 (only in hourly scale)
- hum: Normalized humidity. The values are divided to 100 (max)
- windspeed: Normalized wind speed. The values are divided to 67 (max)
- casual: count of casual users
- registered: count of registered users
- cnt: count of total rental bikes including both casual and registered


# **Data Cleaning**

In [4]:
# Checking columns names
df_day.columns

In [5]:
df_hour.columns

In [6]:
# Size/Shape
df_day.shape, df_hour.shape

In [7]:
# Columns info
df_day.info()

In [8]:
df_hour.info()

In [9]:
# Checking missing values
df_day.isnull().sum()

In [10]:
df_hour.isnull().sum()

In [11]:
# Checking duplicate values
df_day.duplicated().sum()

In [12]:
df_hour.duplicated().sum()

# **Dataset Describe**

In [13]:
# Discribe Numerical Column
df_hour.describe(include=[np.number])

# **Exploratory Data Analysis**

In [14]:
df_season = df_day.groupby('season')['cnt'].sum()

df_season.index = df_season.index.map({
    1: 'Winter',
    2: 'Spring',
    3: 'Summer',
    4: 'Fall'
})
plt.pie(
    df_season,
    labels=df_season.index,
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Seasonal Bike Rental Distribution')
plt.show()

In [15]:
df_mnth = df_day.groupby('mnth')['cnt'].sum().reset_index()

plt.figure(figsize=(10, 6))
a2 = sns.barplot(
        data=df_mnth,
        x='mnth',
        y='cnt'
    )
a2.bar_label(a2.containers[0],padding = 2)
plt.title('Monthly Bike Rental Distribution')
plt.xlabel('Month')
plt.ylabel('count')
plt.show()

In [16]:
df_weekday = df_day.groupby('weekday')['cnt'].sum().reset_index()

plt.figure(figsize=(10,5))
a2 = sns.lineplot(
    data=df_weekday,
    x='weekday',
    y='cnt',
    marker='o'
)

for i in range(len(df_weekday)):
    plt.text(
        df_weekday['weekday'][i],
        df_weekday['cnt'][i] + 500,
        df_weekday['cnt'][i]
    )

plt.title('Weekly Bike Rental Trends')
plt.xlabel('Day of the Week')
plt.ylabel('Total Bike Rentals')
plt.show()

In [17]:
df_hr = df_hour.groupby('hr')['cnt'].sum().reset_index()

plt.figure(figsize=(10, 6))
a2 = sns.barplot(
        data=df_hr,
        x='hr',
        y='cnt'
    )
a2.bar_label(a2.containers[0],padding = 2)
plt.title('Hourly Bike Rental Distribution')
plt.xlabel('Hour')
plt.ylabel('count')
plt.show()

# **Insights**

Summer records the highest number of bike rentals, indicating stronger customer demand during this season.

The bike rental demand is significantly higher from June to September compared to the other months, indicating increased customer activity during this period.

Bike rental activity is highest on the 4th and 5th days of the week, indicating increased customer demand on these days compared to the rest of the week.

The bike rental demand peaks between June and September, particularly on the 4th and 5th days of the week. Additionally, the highest rental activity occurs during the evening hours, between 4:00 PM and 7:00 PM. These periods represent peak demand, suggesting an opportunity to improve bike availability and operational planning during high-traffic hours.

# **Model Creation and Evaluation**

In [18]:
df_hour.head()

In [19]:
X=df_hour[['season','mnth','hr','weekday','weathersit','temp','atemp','hum','windspeed']] 
y=df_hour['cnt']

In [20]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size= 0.2,random_state=12)

In [21]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

In [23]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, min_samples_split=10, random_state=12),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_split=10, random_state=12),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=12)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    print(f"{name} Testing Accuracy : {model.score(X_test, y_test):.3f}")
    print(f"{name} Training Accuracy : {model.score(X_train, y_train):.3f}")
    print(f"{name} RMSE : {rmse:.2f}\n")


# **Model Selection**


*    The Gradient Boosting model demonstrates predictive performance compared to the other evaluated models, achieving high accuracy and a lower RMSE, making it a suitable choice for the given dataset.
*    The Linear Regression model having lower predictive accuracy and overall performance than the other regression models, indicating that it is not the most suitable choice for the given dataset.
*    The Decision Tree and Random Forest models deliver satisfactory predictive results, positioning them as intermediate-performing models among the regression algorithms evaluated.

---
# **Thank You**